# SCC: OpenMP vs CUDA Benchmark (Real SuiteSparse Graphs)

This notebook downloads **real graphs from the SuiteSparse Matrix Collection**,
the same kind the original OpenMP benchmark used, and runs both OpenMP and CUDA
on them for comparison.

**Requires GPU runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
# @title 1. Clone repo and check environment
import os, sys, subprocess, time, re, urllib.request, tarfile, glob

REPO_URL = "https://github.com/ShashwaTTrigunayaT/DynamicGraphs_SCC.git"
REPO_DIR = "/content/DynamicGraphs_SCC"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only  # get latest changes
%cd {REPO_DIR}

!echo "=== GPU Check ==="
!nvidia-smi 2>&1 | head -10
!echo "=== CUDA ==="
!nvcc --version 2>&1
!echo "=== CPUs ==="
!nproc

In [ ]:
# @title 2. Fix CUDA arch and build both binaries
# Fix Makefile for Colab T4 (sm_75)
with open("src_CUDA/Makefile", "r") as f:
    content = f.read()
content = content.replace("-arch=sm_70", "-arch=sm_75")
with open("src_CUDA/Makefile", "w") as f:
    f.write(content)
print("Patched Makefile: sm_70 → sm_75")

# Build OpenMP
print("\n=== Building OpenMP binary ===")
%cd /content/DynamicGraphs_SCC/src
!make -s clean 2>&1; make 2>&1 | tail -10
assert os.path.exists("../scc"), "OpenMP build failed!"
print("✓ OpenMP binary built")

# Build CUDA
print("\n=== Building CUDA binary ===")
%cd /content/DynamicGraphs_SCC/src_CUDA
!make -s clean 2>&1; make 2>&1 | tail -10
assert os.path.exists("scc_cuda"), "CUDA build failed!"
print("✓ CUDA binary built")

BINARY_OMP  = "/content/DynamicGraphs_SCC/scc"
BINARY_CUDA = "/content/DynamicGraphs_SCC/src_CUDA/scc_cuda"

In [ ]:
# @title 3. Download real graphs + generate synthetic large graphs

# ---- Real graphs from SuiteSparse HB collection ----
GRAPHS = [
    ("HB", "west0067", "67 nodes, 207 edges — chemical engineering"),
    ("HB", "dwt_72",   "72 nodes, 276 edges — structural"),
    ("HB", "can_229",  "229 nodes, 774 edges — structural"),
    ("HB", "ash958",   "958 nodes, 3,068 edges — chemical engineering"),
    ("HB", "mbeacxc",  "496 nodes, 22,710 edges — economics"),
    ("HB", "lshp1009", "1,009 nodes, 3,597 edges — structural"),
    ("HB", "bcsstk01", "48 nodes, 400 edges — stiffness matrix"),
]

# ---- Synthetic large graphs (generated locally, no download) ----
SYNTHETIC_GRAPHS = [
    ("synth_100k",  100000,   500000,  "100K nodes, 500K edges"),
]

datasets_dir = "/content/suite_sparse_graphs"
os.makedirs(datasets_dir, exist_ok=True)

# ---- Helper: download .mtx from SuiteSparse ----
def download_mtx(group, name):
    graph_dir = os.path.join(datasets_dir, name)
    os.makedirs(graph_dir, exist_ok=True)
    mtx_file = os.path.join(graph_dir, f"{name}.mtx")
    if os.path.exists(mtx_file): return mtx_file
    url = f"https://sparse.tamu.edu/MM/{group}/{name}.tar.gz"
    tar_path = f"/tmp/{name}.tar.gz"
    print(f"  Downloading {group}/{name}...", end=" ", flush=True)
    try:
        urllib.request.urlretrieve(url, tar_path)
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=graph_dir)
        mtx_files = glob.glob(os.path.join(graph_dir, "**", "*.mtx"), recursive=True)
        if mtx_files:
            import shutil
            shutil.move(mtx_files[0], mtx_file)
            subdir = os.path.dirname(mtx_files[0])
            if subdir and subdir != graph_dir:
                try: shutil.rmtree(subdir)
                except: pass
        print("done")
        return mtx_file
    except Exception as e:
        print(f"FAILED: {e}")
        return None

# ---- Helper: convert .mtx to edge-list ----
def mtx_to_edge_list(mtx_path, output_path):
    """Convert .mtx to refined_edges.txt (1-indexed edge list)."""
    edges = []
    max_v = 0
    header_read = False
    with open(mtx_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('%'): continue
            parts = line.split()
            if not header_read:
                header_read = True  # skip header row
                continue
            if len(parts) < 2: continue  # skip malformed lines
            u = int(parts[0]) - 1  # 1-indexed → 0-indexed
            v = int(parts[1]) - 1
            if u != v:  # skip self-loops
                edges.append((u, v))
                max_v = max(max_v, u, v)
    with open(output_path, 'w') as f:
        for u, v in edges:
            f.write(f"{u+1} {v+1}\n")  # back to 1-indexed
    return max_v + 1, len(edges)

# ---- Helper: generate synthetic graph ----
def generate_synthetic_graph(path, n_nodes, n_edges, seed=42):
    import random
    random.seed(seed)
    written = 0
    with open(path, 'w') as f:
        for _ in range(n_edges * 2):
            u = random.randint(1, n_nodes)
            v = random.randint(1, n_nodes)
            if u != v:
                f.write(f"{u} {v}\n")
                written += 1
                if written >= n_edges: break
    return n_nodes, written

# ---- Download real graphs ----
graph_info = []
print("Downloading real graphs from SuiteSparse...")
for group, name, desc in GRAPHS:
    print(f"  {group}/{name}:", end=" ", flush=True)
    mtx = download_mtx(group, name)
    if mtx:
        out_path = os.path.join(datasets_dir, name, "refined_edges.txt")
        n, m = mtx_to_edge_list(mtx, out_path)
        graph_info.append((name, n, m, desc, out_path))
        print(f"    {name}: {n} nodes, {m} edges")

# ---- Generate synthetic large graphs ----
print("Generating synthetic large graphs...")
synthetic_dir = os.path.join(datasets_dir, "synthetic")
os.makedirs(synthetic_dir, exist_ok=True)
for name, n_nodes, m_edges, desc in SYNTHETIC_GRAPHS:
    out_path = os.path.join(synthetic_dir, f"{name}_edges.txt")
    if os.path.exists(out_path):
        n, m = n_nodes, m_edges
    else:
        n, m = generate_synthetic_graph(out_path, n_nodes, m_edges)
    graph_info.append((name, n, m, desc, out_path))
    print(f"    {name}: {n} nodes, {m} edges")

print(f"Total: {len(graph_info)} graphs ready")

In [ ]:
# @title 4. Benchmark runner
def run_and_parse(binary, graph, threads, method, timeout=600):
    """Run binary and return (runtime_ms, scc_count, full_output)."""
    result = subprocess.run([binary, graph, str(threads), str(method)],
                           capture_output=True, text=True, timeout=timeout)
    output = result.stdout + result.stderr
    
    runtime = scc = None
    for line in output.split('\n'):
        m = re.search(r'running_time\(ms\)=([0-9.]+)', line)
        if m: runtime = float(m.group(1))
        m = re.search(r'Total # SCCs = (\d+)', line)
        if m: scc = int(m.group(1))
    return runtime, scc, output

all_results = []

for name, n_nodes, m_edges, desc, graph_path in graph_info:
    print(f"\n{'='*70}")
    print(f"Graph: {name}  ({desc})")
    print(f"{'='*70}")
    
    row = {"name": name, "nodes": n_nodes, "edges": m_edges}
    
    # === OpenMP Method 0 (Baseline: Trim1 + FW-BW BFS) ===
    omp0_rt, omp0_scc, omp0_out = run_and_parse(BINARY_OMP, graph_path, 8, 0)
    row["omp0_rt"] = omp0_rt
    row["omp0_scc"] = omp0_scc
    for line in omp0_out.split('\n'):
        if 'running_time' in line or 'Total # SCCs' in line:
            print(f"  [OMP0]  {line.strip()}")
    
    # === OpenMP Method 1 (Global FB + Trim1 + FW-BW DFS) ===
    omp1_rt, omp1_scc, omp1_out = run_and_parse(BINARY_OMP, graph_path, 8, 1)
    row["omp1_rt"] = omp1_rt
    row["omp1_scc"] = omp1_scc
    for line in omp1_out.split('\n'):
        if 'running_time' in line or 'Total # SCCs' in line:
            print(f"  [OMP1]  {line.strip()}")
    
    # === OpenMP Method 2 (Full pipeline) ===
    omp2_rt, omp2_scc, omp2_out = run_and_parse(BINARY_OMP, graph_path, 8, 2)
    row["omp2_rt"] = omp2_rt
    row["omp2_scc"] = omp2_scc
    for line in omp2_out.split('\n'):
        if 'running_time' in line or 'Total # SCCs' in line:
            print(f"  [OMP2]  {line.strip()}")
    
    # === CUDA Method 0 (Baseline: Trim1 + FW-BW BFS) ===
    cuda0_rt, cuda0_scc, cuda0_out = run_and_parse(BINARY_CUDA, graph_path, 8, 0)
    row["cuda0_rt"] = cuda0_rt
    row["cuda0_scc"] = cuda0_scc
    for line in cuda0_out.split('\n'):
        if 'running_time' in line or 'Total # SCCs' in line:
            print(f"  [CUDA0] {line.strip()}")
    
    # === CUDA Method 1 (Global FB + Trim1 + FW-BW DFS) ===
    cuda1_rt, cuda1_scc, cuda1_out = run_and_parse(BINARY_CUDA, graph_path, 8, 1)
    row["cuda1_rt"] = cuda1_rt
    row["cuda1_scc"] = cuda1_scc
    for line in cuda1_out.split('\n'):
        if 'running_time' in line or 'Total # SCCs' in line:
            print(f"  [CUDA1] {line.strip()}")
    
    # === CUDA Method 2 (Full pipeline) ===
    cuda2_rt, cuda2_scc, cuda2_out = run_and_parse(BINARY_CUDA, graph_path, 8, 2)
    row["cuda2_rt"] = cuda2_rt
    row["cuda2_scc"] = cuda2_scc
    for line in cuda2_out.split('\n'):
        if 'running_time' in line or 'Total # SCCs' in line:
            print(f"  [CUDA2] {line.strip()}")
    
    all_results.append(row)

In [ ]:
# @title 5. Comparison Table
print("=" * 160)
h  = f"{'Graph':<15s} {'Nodes':<8s} {'Edges':<10s} | "
h += f"{'OMP0(ms)':<10s} {'SCC':<7s} | "
h += f"{'OMP1(ms)':<10s} {'SCC':<7s} | "
h += f"{'OMP2(ms)':<10s} {'SCC':<7s} | "
h += f"{'C0(ms)':<10s} {'SCC':<7s} | "
h += f"{'C1(ms)':<10s} {'SCC':<7s} | "
h += f"{'C2(ms)':<10s} {'SCC':<7s} | "
h += f"{'C0≈OMP0':<9s} {'C1≈OMP1':<9s} {'C2≈OMP2'}"
print(h)
print("-" * 160)

def fmt_ms(v):
    if v is not None and v > 0: return f"{v:.2f}"
    return "FAIL"
def fmt_scc(v):
    if v is not None and v >= 0: return str(v)
    return "?"
def match_str(omp_scc, cuda_scc):
    if omp_scc is None or cuda_scc is None or omp_scc < 0 or cuda_scc < 0:
        return "?"
    if omp_scc == cuda_scc: return "✓"
    return f"✗(d={cuda_scc-omp_scc})"

for r in all_results:
    name = r['name']
    n = r['nodes']
    m = r['edges']
    o0_rt = r.get('omp0_rt', -1)
    o0_sc = r.get('omp0_scc', -1)
    o1_rt = r.get('omp1_rt', -1)
    o1_sc = r.get('omp1_scc', -1)
    o2_rt = r.get('omp2_rt', -1)
    o2_sc = r.get('omp2_scc', -1)
    c0_rt = r.get('cuda0_rt', -1)
    c0_sc = r.get('cuda0_scc', -1)
    c1_rt = r.get('cuda1_rt', -1)
    c1_sc = r.get('cuda1_scc', -1)
    c2_rt = r.get('cuda2_rt', -1)
    c2_sc = r.get('cuda2_scc', -1)
    
    s  = f"{name:<15s} {n:<8d} {m:<10d} | "
    s += f"{fmt_ms(o0_rt):<10s} {fmt_scc(o0_sc):<7s} | "
    s += f"{fmt_ms(o1_rt):<10s} {fmt_scc(o1_sc):<7s} | "
    s += f"{fmt_ms(o2_rt):<10s} {fmt_scc(o2_sc):<7s} | "
    s += f"{fmt_ms(c0_rt):<10s} {fmt_scc(c0_sc):<7s} | "
    s += f"{fmt_ms(c1_rt):<10s} {fmt_scc(c1_sc):<7s} | "
    s += f"{fmt_ms(c2_rt):<10s} {fmt_scc(c2_sc):<7s} | "
    s += f"{match_str(o0_sc, c0_sc):<9s} {match_str(o1_sc, c1_sc):<9s} {match_str(o2_sc, c2_sc)}"
    print(s)

print("-" * 160)
print("OMP0 / C0 = Baseline: Trim1 + FW-BW BFS")
print("OMP1 / C1 = Trim1 + Global FB + Trim1 + FW-BW DFS")
print("OMP2 / C2 = Full pipeline: Trim1 + Global FB + Trim1/2 + WCC + FW-BW DFS")
print("✓ = SCC count matches exactly between OMP and CUDA")
print("✗(d=N) = CUDA found N fewer/more SCCs than OMP")

In [ ]:
# @title 6. Download as Excel spreadsheet
import pandas as pd
from google.colab import files

rows = []
for r in all_results:
    def g(k): return r.get(k, -1)
    rows.append({
        "Graph": r["name"],
        "Nodes": r["nodes"],
        "Edges": r["edges"],
        "OMP0 (ms)": round(g("omp0_rt"), 2) if g("omp0_rt") > 0 else "FAIL",
        "OMP0 SCC": g("omp0_scc"),
        "OMP1 (ms)": round(g("omp1_rt"), 2) if g("omp1_rt") > 0 else "FAIL",
        "OMP1 SCC": g("omp1_scc"),
        "OMP2 (ms)": round(g("omp2_rt"), 2) if g("omp2_rt") > 0 else "FAIL",
        "OMP2 SCC": g("omp2_scc"),
        "CUDA0 (ms)": round(g("cuda0_rt"), 2) if g("cuda0_rt") > 0 else "FAIL",
        "CUDA0 SCC": g("cuda0_scc"),
        "CUDA1 (ms)": round(g("cuda1_rt"), 2) if g("cuda1_rt") > 0 else "FAIL",
        "CUDA1 SCC": g("cuda1_scc"),
        "CUDA2 (ms)": round(g("cuda2_rt"), 2) if g("cuda2_rt") > 0 else "FAIL",
        "CUDA2 SCC": g("cuda2_scc"),
        "M0 Match": match_str(g("omp0_scc"), g("cuda0_scc")),
        "M1 Match": match_str(g("omp1_scc"), g("cuda1_scc")),
        "M2 Match": match_str(g("omp2_scc"), g("cuda2_scc")),
    })

df = pd.DataFrame(rows)

xlsx_path = "/content/scc_benchmark_results.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Comparison", index=False)
    ws = writer.sheets["Comparison"]
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = max_len + 3

print(f"Excel exported: {xlsx_path}")
print("Downloading...")
files.download(xlsx_path)
print("✓ Done! Open the .xlsx in Excel or Google Sheets.")